<a href="https://colab.research.google.com/github/Rheaxu/story-converter-ai/blob/main/K5ReadingStoryConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1.1: Install Dependencies
Install PyMuPDF for PDF processing and edge-tts for high-quality neural voice generation.

In [ ]:
### Step 1.1: Install Dependencies
# Install PyMuPDF for PDF processing and edge-tts for high-quality voice generation.
!pip install pymupdf edge-tts

# Step 2a: Public Story Converter Core Logic
This tool extracts text and images from your PDFs and generates a standard interactive HTML website for public hosting.

In [ ]:
### Step 2a: Public Story Converter Core Logic (With Globe-Toggleable Translations)

import fitz
import os
import base64
import json
import asyncio
import ipywidgets as widgets
from IPython.display import display
import edge_tts

def extract_pdf_data(pdf_path):
    doc = fitz.open(pdf_path)
    story_data = []
    for page in doc:
        page_rect = page.rect
        page_content = {
            "text_blocks": [],
            "word_bank": [],
            "images": [],
            "dimensions": [page_rect.width, page_rect.height]
        }

        img_infos = page.get_image_info(hashes=False, xrefs=True)
        for info in img_infos:
            bbox = info['bbox']
            try:
                pix = page.get_pixmap(
                    clip=bbox,
                    alpha=True,
                    matrix=fitz.Matrix(2, 2),
                    annots=False
                )
                img_data = pix.tobytes(
                    "png"
                )
                img_base64 = base64.b64encode(img_data).decode('utf-8')
                page_content["images"].append({
                    "base64": img_base64,
                    "bbox": bbox
                })
                pix = None
            except Exception as e:
                print(f"Skipping image at {bbox} due to: {e}")

        blocks = page.get_text("dict")[
            "blocks"
        ]
        for b in blocks:
            if b["type"] == 0:
                for l in b["lines"]:
                    for s in l["spans"]:
                        clean_text = s["text"].strip()
                        if not clean_text: continue
                        x0, y0, x1, y1 = s["bbox"]

                        is_in_potential_word_bank_area = (y0 < page_rect.height * 0.25) and (x0 > page_rect.width * 0.30)
                        is_short_text = (len(clean_text.split()) <= 5)
                        is_footer = (y0 < page_rect.height * 0.08) or (y0 > page_rect.height * 0.92)
                        is_reader_label = any(x in clean_text.lower() for x in ["level", "reader"])

                        data = {
                            "text": s["text"],
                            "bbox": s["bbox"],
                            "size": s["size"],
                            "font": s["font"],
                            "is_footer": is_footer or is_reader_label
                        }

                        if is_in_potential_word_bank_area and is_short_text and not is_footer and not is_reader_label:
                            words_in_span = clean_text.split()
                            for word in words_in_span:
                                clean_w = word.strip(",.?!;: ")
                                if clean_w:
                                    if clean_w not in page_content["word_bank"]:
                                        page_content["word_bank"].append(clean_w)
                        else:
                            page_content["text_blocks"].append(data)
        story_data.append(page_content)
    return story_data

async def generate_audio(text, output_path, voice="en-US-GuyNeural"):
    communicate = edge_tts.Communicate(text, voice, rate="-20%")
    await communicate.save(output_path)

def generate_html(pdf_name, story_data, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    audio_dir_name = f"audio_{pdf_name.replace('.pdf', '')}"
    audio_dir_path = os.path.join(output_folder, audio_dir_name)
    os.makedirs(audio_dir_path, exist_ok=True)

    pages_html = []
    sentence_count = 0
    loop = asyncio.get_event_loop()
    unique_words = []
    vocab_html = '<div class="word-bank"><h2>Word Bank</h2>'
    for p in story_data:
        for txt in p["word_bank"]:
            if txt not in unique_words:
                unique_words.append(txt)
                sentence_count += 1
                audio_file = f"vocab_{sentence_count}.mp3"
                audio_path = os.path.join(audio_dir_path, audio_file)
                if loop.is_running(): loop.create_task(generate_audio(txt, audio_path))
                else: asyncio.run(generate_audio(txt, audio_path))
                translation = translations_db.get(txt, "N/A")
                vocab_html += f"""<div class="word-wrapper">
                    <div class="word-item" onclick="playAudio(this, '{audio_dir_name}/{audio_file}')">{txt}</div>
                    <div class="translation-box" onclick="toggleTranslation(this, '{translation}')">🌐</div>
                </div>"""

    is_first_sentence = True
    for p in story_data:
        w, h = p["dimensions"]
        scale = 2.0
        original_height_scaled = h * scale

        calculated_elements = []
        y_offsets = {}
        max_y_reached = original_height_scaled

        for txt in p["text_blocks"]:
            x0, y0, x1, y1 = txt["bbox"]
            if txt["is_footer"]:
                # If it's a footer or header at the very top, keep its height exact
                # Let's distinguish top headers (very top) and bottom footers:
                is_header_top = (y0 * scale < original_height_scaled * 0.1)
                if is_header_top:
                    calculated_elements.append({
                        "is_footer": False,
                        "x0": x0 * scale,
                        "top": y0 * scale, # Preserve exact top distance
                        "class": "sentence-wrapper footer-text",
                        "audio_file": None,
                        "text": txt["text"],
                        "translation": ""
                    })
                else:
                    dist_from_bottom = original_height_scaled - (y0 * scale)
                    calculated_elements.append({
                        "is_footer": True,
                        "x0": x0 * scale,
                        "dist_from_bottom": dist_from_bottom,
                        "size": txt["size" ] * scale * 0.5,
                        "text": txt["text"]
                    })
            else:
                col_key = round(x0 * scale, -1)
                curr_top = y0 * scale

                # If it is physically within the top header zone (top 10%), preserve original top to align with other headers
                is_header_top = (curr_top < original_height_scaled * 0.1)
                if not is_header_top:
                    if col_key in y_offsets:
                        curr_top = max(curr_top, y_offsets[col_key] + 130)
                    y_offsets[col_key] = curr_top

                needed_room = curr_top + 260
                if needed_room > max_y_reached:
                    max_y_reached = needed_room

                sentence_count += 1
                audio_file = f"s_{sentence_count}.mp3"
                audio_path = os.path.join(audio_dir_path, audio_file)
                if loop.is_running(): loop.create_task(generate_audio(txt["text"].strip(), audio_path))
                else: asyncio.run(generate_audio(txt["text"].strip(), audio_path))
                cls = "sentence story-title" if is_first_sentence else "sentence"
                translation = translations_db.get(txt["text"].strip(), "N/A")

                calculated_elements.append({
                    "is_footer": False,
                    "x0": x0 * scale,
                    "top": curr_top,
                    "class": cls,
                    "audio_file": audio_file,
                    "text": txt["text"],
                    "translation": translation
                })
                is_first_sentence = False

        page_html = f'<div class="page" style="width: {w * scale}px; height: {max_y_reached}px;">'
        for img in p["images"]:
            ix0, iy0, ix1, iy1 = img["bbox"]
            page_html += f'<img src="data:image/png;base64,{img["base64"]}" style="position:absolute; left:{ix0 * scale}px; top:{iy0 * scale}px; width:{(ix1-ix0) * scale}px; height:{(iy1-iy0) * scale}px;">'

        for el in calculated_elements:
            if el["is_footer"]:
                new_top = max_y_reached - el["dist_from_bottom"]
                style = f"left:{el['x0']}px; top:{new_top}px; font-size:{el['size']}px;"
                page_html += f'<div class="absolute-element footer-text" style="{style}">{el["text"]}</div>'
            else:
                style = f"left:{el['x0']}px; top:{el['top']}px;"
                if el.get("audio_file"):
                    page_html += f"""<div class="absolute-element sentence-wrapper" style="{style}">
                        <div class="{el['class']}" onclick="playAudio(this, '{audio_dir_name}/{el['audio_file']}')">{el['text']}</div>
                        <div class="translation-box" onclick="toggleTranslation(this, '{el['translation']}')">🌐</div>
                    </div>"""
                else:
                    # Decorative / static top headers that do not require audio play links
                    page_html += f'<div class="absolute-element" style="{style}; font-size: 24px; font-weight: bold; color: #555;">{el["text"]}</div>'
        page_html += '</div>'
        pages_html.append(page_html)

    html_template = """<html><head><meta charset="utf-8"><style>
        body { background: #f0f2f5; display: flex; flex-direction: column; align-items: center; padding: 20px; margin: 0; font-family: 'Segoe UI', sans-serif; }
        .word-bank { background: #fff; width: 90%; max-width: 950px; padding: 20px; margin: 20px 0; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); display: flex; flex-wrap: wrap; gap: 12px; justify-content: center; border: 4px solid #4a90e2; }
        .word-bank h2 { width: 100%; text-align: center; color: #4a90e2; font-size: 40px; font-weight: 800; }
        .word-wrapper { display: flex; flex-direction: column; align-items: center; gap: 6px; }
        .word-item { padding: 10px 20px; background: #f8f9fa; color: #4a90e2; border-radius: 8px; font-weight: bold; cursor: pointer; border: 2px solid #4a90e2; font-size: 2rem; }
        .translation-box { font-size: 1.2rem; color: #e67e22; font-weight: bold; background: #fff3cd; border: 1px solid #ffeeba; border-radius: 6px; padding: 6px 12px; text-align: center; cursor: pointer; display: block; margin-top: 8px; z-index: 100; position: relative; max-width: max-content; }
        .page { background: white; position: relative; margin-bottom: 40px; box-shadow: 0 5px 15px rgba(0,0,0,0.2); overflow: hidden; }
        .absolute-element { position: absolute; transform-origin: 0 0; line-height: 1.2; }
        .sentence-wrapper { display: flex; flex-direction: column; align-items: flex-start; z-index: 10; padding: 8px; border-radius: 8px; transition: background 0.3s; }
        .sentence { cursor: pointer; font-weight: 500; font-size: 50px !important; border-radius: 8px; display: inline-block; white-space: nowrap; }
        .story-title { font-size: 70px !important; font-weight: 700 !important; }
        .footer-text { z-index: 5; opacity: 0.6; pointer-events: none; white-space: nowrap; }
        .playing { background: rgba(74, 144, 226, 0.3) !important; }
        img { z-index: 1; }
    </style><script>
        function playAudio(el, f){ const audio = new Audio(f); el.classList.add('playing'); audio.onended = () => el.classList.remove('playing'); audio.play(); }
        function toggleTranslation(box, transText) {
            if (box.innerHTML === '🌐') {
                box.innerHTML = transText;
            } else {
                box.innerHTML = '🌐';
            }
        }
    </script></head><body>"""

    if unique_words: html_template += vocab_html + '</div>'
    for p_html in pages_html:
        html_template += p_html
    html_template += "</body></html>"
    return html_template

pdf_list = [f for f in os.listdir(".") if f.endswith(".pdf")]
dropdown = widgets.Dropdown(options=pdf_list if pdf_list else ["No PDFs found"], description="Select PDF:")
folder_input = widgets.Text(value="final_output", description="Output Folder:")
button = widgets.Button(description="Convert", button_style="primary")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        if dropdown.value == "No PDFs found": return
        print(f"Processing {dropdown.value}...")
        try:
            data = extract_pdf_data(dropdown.value)
            html = generate_html(dropdown.value, data, folder_input.value)
            output_path = os.path.join(folder_input.value, dropdown.value.replace(".pdf", ".html"))
            with open(output_path, "w", encoding='utf-8') as f:
                f.write(html)
            print(f"Success! Interactive story generated at: {output_path}")
        except Exception as e:
            print(f"Error: {e}")

button.on_click(on_click)
display(dropdown, folder_input, button, output)

### Step 2.1: Interactive Translation Manager
Now that you have analyzed the story, use this interactive control panel to load, modify, and manage translation mappings.

1. Click **Scan PDFs for Words/Lines** to pull all parsed lines into the translator workspace.
2. Translate any words or sentences by typing directly into the textboxes below.
3. Click **Save JSON** to save your work before converting.

In [ ]:
import json
import os
import fitz
import ipywidgets as widgets
from IPython.display import display

# Global translation dictionary shared across the notebook
translations_db = globals().get('translations_db', {})

trans_json_input = widgets.Text(value="translations.json", description="JSON File:")
load_trans_btn = widgets.Button(description="Load JSON", button_style="info")
save_trans_btn = widgets.Button(description="Save JSON", button_style="success")
apply_trans_btn = widgets.Button(description="Apply Translations", button_style="primary")

# Manual entry fields
manual_word_input = widgets.Text(description="Original:", placeholder="e.g., Cat")
manual_trans_input = widgets.Text(description="Translation:", placeholder="e.g., Gato")
add_trans_btn = widgets.Button(description="Add / Update", button_style="primary")

scan_pdf_btn = widgets.Button(description="Scan PDFs for Words/Lines", button_style="warning")
refresh_matrix_btn = widgets.Button(description="Refresh Matrix View", button_style="")

trans_list_container = widgets.VBox()
trans_status = widgets.Output()

def update_textbox_list():
    """Refreshes individual interactive text boxes for each word/sentence mapping."""
    textboxes = []
    if not translations_db:
        trans_list_container.children = [widgets.HTML("<p style='color: gray; margin-left: 20px;'>No translation keys registered yet. Upload a PDF and click 'Scan PDFs' or add entries manually to populate this list.</p>")]
        return

    # Generate an interactive input box for each original word or sentence
    for orig, trans in sorted(translations_db.items()):
        tb = widgets.Text(value=trans, description=orig, style={'description_width': 'initial'}, placeholder=f"Type translation for: {orig}")
        # Create a closure to bind the current key to its change handler
        def make_change_handler(o):
            return lambda change: translations_db.update({o: change['new']})
        tb.observe(make_change_handler(orig), names='value')
        textboxes.append(tb)
    trans_list_container.children = textboxes

def scan_workspace_pdfs(silent=False):
    """Extracts unique word-bank words and sentences from any PDF in the current directory and adds them to DB."""
    pdf_files = [f for f in os.listdir(".") if f.endswith(".pdf")]
    if not pdf_files:
        if not silent:
            print("No PDFs found in current directory. Please upload your story PDF first!")
        return

    count_added = 0
    for pdf in pdf_files:
        try:
            doc = fitz.open(pdf)
            for page in doc:
                page_rect = page.rect
                text_dict = page.get_text("dict")["blocks"]
                for b in text_dict:
                    if b["type"] == 0:
                        for l in b["lines"]:
                            for s in l["spans"]:
                                text_stripped = s["text"].strip()
                                if not text_stripped:
                                    continue
                                x0, y0, x1, y1 = s["bbox"]
                                is_in_potential_word_bank_area = (y0 < page_rect.height * 0.25) and (x0 > page_rect.width * 0.30)
                                is_short_text = (len(text_stripped.split()) <= 5)
                                is_footer = (y0 < page_rect.height * 0.08) or (y0 > page_rect.height * 0.92)
                                is_reader_label = any(x in text_stripped.lower() for x in ["level", "reader"])

                                if is_in_potential_word_bank_area and is_short_text and not is_footer and not is_reader_label:
                                    for word in text_stripped.split():
                                        clean_w = word.strip(",.?!;: ")
                                        if clean_w and clean_w not in translations_db:
                                            translations_db[clean_w] = ""
                                            count_added += 1
                                else:
                                    if not is_footer and not is_reader_label and text_stripped not in translations_db:
                                        translations_db[text_stripped] = ""
                                        count_added += 1
        except Exception as e:
            if not silent:
                print(f"Error scanning {pdf}: {e}")
    if not silent:
        print(f"Scanned PDFs! Added {count_added} new items to translation workspace list.")
    update_textbox_list()

def load_translations_from_file(filepath):
    global translations_db
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Explicitly clear db first to drop old keys
                translations_db.clear()
                translations_db.update(data.get("translations", {}))
            print(f"Loaded {len(translations_db)} translations from {filepath}.")
            update_textbox_list()
        except Exception as e:
            print(f"Error loading JSON file: {e}")
    else:
        print(f"File {filepath} not found. Starting with empty/current database.")

def save_translations_to_file(filepath):
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({"translations": translations_db}, f, ensure_ascii=False, indent=2)
        print(f"Saved {len(translations_db)} translations to {filepath}")
    except Exception as e:
        print(f"Error saving file: {e}")

def on_load_clicked(b):
    with trans_status:
        trans_status.clear_output()
        load_translations_from_file(trans_json_input.value)

def on_save_clicked(b):
    with trans_status:
        trans_status.clear_output()
        save_translations_to_file(trans_json_input.value)

def on_add_clicked(b):
    with trans_status:
        trans_status.clear_output()
        orig = manual_word_input.value.strip()
        trans = manual_trans_input.value.strip()
        if orig:
            translations_db[orig] = trans
            print(f"Added: '{orig}' -> '{trans}'")
            manual_word_input.value = ""
            manual_trans_input.value = ""
            update_textbox_list()
        else:
            print("Please provide an original text entry.")

def on_scan_clicked(b):
    with trans_status:
        trans_status.clear_output()
        scan_workspace_pdfs()

def on_refresh_clicked(b):
    with trans_status:
        trans_status.clear_output()
        update_textbox_list()
        print("Refreshed layout.")

def on_apply_clicked(b):
    with trans_status:
        trans_status.clear_output()
        save_translations_to_file(trans_json_input.value)
        print("Triggering Step 2a conversion to apply translations...")
        if 'on_click' in globals() and 'button' in globals():
            try:
                on_click(None)
                print("Step 2a converter successfully run automatically.")
            except Exception as e:
                print(f"Could not automatically run converter: {e}")
        else:
            print("Step 2a converter elements could not be found in active memory. Please make sure you have executed Step 2a's code cell first.")

load_trans_btn.on_click(on_load_clicked)
save_trans_btn.on_click(on_save_clicked)
add_trans_btn.on_click(on_add_clicked)
scan_pdf_btn.on_click(on_scan_clicked)
refresh_matrix_btn.on_click(on_refresh_clicked)
apply_trans_btn.on_click(on_apply_clicked)

try:
    scan_workspace_pdfs(silent=True)
except Exception:
    pass

display(
    widgets.VBox([
        widgets.HTML("<h3>Interactive Translation UI Control Panel</h3>"),
        widgets.HTML("<h4>Load existing dictionary:</h4>"),
        widgets.HBox([trans_json_input, load_trans_btn]),
        widgets.HTML("<h4>Automatically extract words/sentences to translate from PDF:</h4>"),
        widgets.HBox([scan_pdf_btn, refresh_matrix_btn]),
        widgets.HTML("<h4>Or type translation manually:</h4>"),
        widgets.HBox([manual_word_input, manual_trans_input, add_trans_btn]),
        widgets.HTML("<h4>Translations Textboxes Matrix (Modify below to update database)</h4>"),
        trans_list_container,
        widgets.HTML("<h4>Apply and Save Mappings:</h4>"),
        widgets.HBox([save_trans_btn, apply_trans_btn]),
        trans_status
    ])
)
update_textbox_list()

# Step 2b: Private Story Converter Core Logic
This version allows you to include a **SAS Token**. The token is appended to audio URLs in the HTML, enabling the interactive features to work even when assets are hosted in private cloud storage.

In [ ]:
### Step 2b: Private Story Converter (Dynamic URL, SAS & Globe Translations Integration)

import fitz, os, base64, asyncio, edge_tts
import ipywidgets as widgets
from IPython.display import display

async def generate_audio(text, output_path, voice="en-US-GuyNeural"):
    """Utility function to generate TTS audio files using edge-tts."""
    communicate = edge_tts.Communicate(text, voice, rate="-20%")
    await communicate.save(output_path)

async def generate_private_story_async(pdf_name, story_data, output_folder):
    story_id = pdf_name.replace('.pdf', '')
    story_dir = os.path.join(output_folder, story_id)
    os.makedirs(story_dir, exist_ok=True)
    audio_dir_name = f"audio_{story_id}"
    audio_dir_path = os.path.join(story_dir, audio_dir_name)
    os.makedirs(audio_dir_path, exist_ok=True)

    # Temporary templates and elements container variables
    pages_html = []
    audio_tasks, unique_words, sentence_count = [], [], 0
    vocab_html = '<div class="word-bank"><h2>Word Bank</h2>'
    for p in story_data:
        for txt in p["word_bank"]:
            if txt not in unique_words:
                unique_words.append(txt)
                sentence_count += 1
                audio_file = f"vocab_{sentence_count}.mp3"
                audio_tasks.append(generate_audio(txt, os.path.join(audio_dir_path, audio_file)))
                translation = translations_db.get(txt, "N/A")
                vocab_html += f"""<div class="word-wrapper">
                    <div class="word-item" onclick="playAudio(this, '{audio_file}')">{txt}</div>
                    <div class="translation-box" onclick="toggleTranslation(this, '{translation}')">🌐</div>
                </div>"""

    # Resolve dynamic size and custom layouts loop
    is_first_sentence = True
    for p in story_data:
        w, h = p["dimensions"]
        scale = 2.0
        original_height_scaled = h * scale

        # First pre-calculate elements and keep track of maximum required vertical height
        calculated_elements = []
        y_offsets = {}
        max_y_reached = original_height_scaled

        for txt in p["text_blocks"]:
            x0, y0, x1, y1 = txt["bbox"]
            if txt["is_footer"]:
                # Record footer original distance from bottom
                is_header_top = (y0 * scale < original_height_scaled * 0.1)
                if is_header_top:
                    calculated_elements.append({
                        "is_footer": False,
                        "x0": x0 * scale,
                        "top": y0 * scale,
                        "class": "sentence-wrapper footer-text",
                        "audio_file": None,
                        "text": txt["text"],
                        "translation": ""
                    })
                else:
                    dist_from_bottom = original_height_scaled - (y0 * scale)
                    calculated_elements.append({
                        "is_footer": True,
                        "x0": x0 * scale,
                        "dist_from_bottom": dist_from_bottom,
                        "size": txt["size"] * scale * 0.5,
                        "text": txt["text"]
                    })
            else:
                col_key = round(x0 * scale, -1)
                curr_top = y0 * scale

                is_header_top = (curr_top < original_height_scaled * 0.1)
                if not is_header_top:
                    if col_key in y_offsets:
                        curr_top = max(curr_top, y_offsets[col_key] + 130)
                    y_offsets[col_key] = curr_top

                # Track total page height expansion requirements (needs room for text + translation box + 100px safety margin)
                needed_room = curr_top + 260
                if needed_room > max_y_reached:
                    max_y_reached = needed_room

                sentence_count += 1
                audio_file = f"s_{sentence_count}.mp3"
                audio_tasks.append(generate_audio(txt["text"].strip(), os.path.join(audio_dir_path, audio_file)))
                cls = "sentence story-title" if is_first_sentence else "sentence"
                translation = translations_db.get(txt["text"].strip(), "N/A")

                calculated_elements.append({
                    "is_footer": False,
                    "x0": x0 * scale,
                    "top": curr_top,
                    "class": cls,
                    "audio_file": audio_file,
                    "text": txt["text"],
                    "translation": translation
                })
                is_first_sentence = False

        page_html = f'<div class="page" style="width: {w * scale}px; height: {max_y_reached}px;">'
        for img in p["images"]:
            ix0, iy0, ix1, iy1 = img["bbox"]
            page_html += f'<img src="data:image/png;base64,{img["base64"]}" style="position:absolute; left:{ix0 * scale}px; top:{iy0 * scale}px; width:{(ix1-ix0) * scale}px; height:{(iy1-iy0) * scale}px;">'

        for el in calculated_elements:
            if el["is_footer"]:
                # Put footer relative to new expanded bottom coordinate
                new_top = max_y_reached - el["dist_from_bottom"]
                style = f"left:{el['x0']}px; top:{new_top}px; font-size:{el['size']}px;"
                page_html += f'<div class="absolute-element footer-text" style="{style}">{el["text"]}</div>'
            else:
                style = f"left:{el['x0']}px; top:{el['top']}px;"
                if el.get("audio_file"):
                    page_html += f"""<div class="absolute-element sentence-wrapper" style="{style}">
                        <div class="{el['class']}" onclick="playAudio(this, '{el['audio_file']}')">{el['text']}</div>
                        <div class="translation-box" onclick="toggleTranslation(this, '{el['translation']}')">🌐</div>
                    </div>"""
                else:
                    page_html += f'<div class="absolute-element" style="{style}; font-size: 24px; font-weight: bold; color: #555;">{el["text"]}</div>'
        page_html += '</div>'
        pages_html.append(page_html)

    html_template = f"""<html><head><meta charset=\"utf-8\"><style>
        body {{ background: #f0f2f5; display: flex; flex-direction: column; align-items: center; padding: 20px; margin: 0; font-family: 'Segoe UI', sans-serif; }}
        .word-bank {{ background: #fff; width: 90%; max-width: 950px; padding: 20px; margin: 20px 0; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); display: flex; flex-wrap: wrap; gap: 12px; justify-content: center; border: 4px solid #4a90e2; }}
        .word-bank h2 {{ width: 100%; text-align: center; color: #4a90e2; font-size: 40px; font-weight: 800; }}
        .word-wrapper {{ display: flex; flex-direction: column; align-items: center; gap: 6px; }}
        .word-item {{ padding: 10px 20px; background: #f8f9fa; color: #4a90e2; border-radius: 8px; font-weight: bold; cursor: pointer; border: 2px solid #4a90e2; font-size: 2rem; }}
        .translation-box {{ font-size: 1.2rem; color: #e67e22; font-weight: bold; background: #fff3cd; border: 1px solid #ffeeba; border-radius: 6px; padding: 6px 12px; text-align: center; cursor: pointer; display: block; margin-top: 8px; z-index: 100; position: relative; max-width: max-content; }}
        .page {{ background: white; position: relative; margin-bottom: 40px; box-shadow: 0 5px 15px rgba(0,0,0,0.2); overflow: hidden; }}
        .absolute-element {{ position: absolute; transform-origin: 0 0; line-height: 1.2; }}
        .sentence-wrapper {{ display: flex; flex-direction: column; align-items: flex-start; z-index: 10; padding: 8px; border-radius: 8px; transition: background 0.3s; }}
        .sentence {{ cursor: pointer; font-weight: 500; font-size: 50px !important; border-radius: 8px; display: inline-block; white-space: nowrap; }}
        .story-title {{ font-size: 70px !important; font-weight: 700 !important; }}
        .footer-text {{ z-index: 5; opacity: 0.6; pointer-events: none; white-space: nowrap; }}
        .playing {{ background: rgba(74, 144, 226, 0.3) !important; }}
        img {{ z-index: 1; }}
    </style><script>
        function playAudio(el, filename){{
            const currentUrl = window.location.href.split('?')[0];
            const baseUrl = currentUrl.substring(0, currentUrl.lastIndexOf('/') + 1);
            const sasToken = window.location.search;
            const audioUrl = baseUrl + "{audio_dir_name}/" + filename + sasToken;
            const audio = new Audio(audioUrl);
            el.classList.add('playing');
            audio.onended = () => el.classList.remove('playing');
            audio.play();
        }}
        function toggleTranslation(box, transText) {{
            if (box.innerHTML === '🌐') {{
                box.innerHTML = transText;
            }} else {{
                box.innerHTML = '🌐';
            }}
        }}
    </script></head><body>"""

    if unique_words:
        html_template += vocab_html + '</div>'

    for p_html in pages_html:
        html_template += p_html

    html_template += "</body></html>"

    if audio_tasks:
        await asyncio.gather(*audio_tasks)

    with open(os.path.join(story_dir, "index.html"), "w", encoding='utf-8') as f:
        f.write(html_template)

    return os.path.join(story_dir, "index.html")

pdf_list_files = [f for f in os.listdir(".") if f.endswith(".pdf")]
dropdown_pa = widgets.Dropdown(options=pdf_list_files if pdf_list_files else ["No PDFs found"], description="Select PDF:")
button_pa = widgets.Button(description="Generate Private Story", button_style="warning")
output_pa = widgets.Output()

def on_click_pa(b):
    with output_pa:
        output_pa.clear_output()
        if dropdown_pa.value == "No PDFs found":
            print("Please upload a PDF first!")
            return
        print(f"Starting conversion for {dropdown_pa.value}...")

        async def run_process():
            try:
                data = extract_pdf_data(dropdown_pa.value)
                result_path = await generate_private_story_async(dropdown_pa.value, data, "private_build")
                print(f"Success! Private website generated at: {result_path}")
            except Exception as e:
                print(f"Error: {e}")

        asyncio.ensure_future(run_process())

button_pa.on_click(on_click_pa)
display(dropdown_pa, button_pa, output_pa)

In [ ]:
### Run Auto-Regeneration with fixed layouts immediately
import asyncio

try:
    # Auto run Public Generation
    print("Auto-regenerating Public HTML layout...")
    data_public = extract_pdf_data("kb13_SoManyLeaves.pdf")
    html_public = generate_html("kb13_SoManyLeaves.pdf", data_public, "final_output")
    with open("final_output/kb13_SoManyLeaves.html", "w", encoding='utf-8') as f:
        f.write(html_public)
    print("Public HTML successfully regenerated at: final_output/kb13_SoManyLeaves.html")
except Exception as e:
    print(f"Could not auto-generate Public Story: {e}")

try:
    # Auto run Private Generation using loop-aware context
    print("\nAuto-regenerating Private HTML layout...")
    loop = asyncio.get_event_loop()
    if loop.is_running():
        loop.create_task(generate_private_story_async("kb13_SoManyLeaves.pdf", data_public, "private_build"))
        print("Private HTML generation task queued successfully in active event loop.")
    else:
        asyncio.run(generate_private_story_async("kb13_SoManyLeaves.pdf", data_public, "private_build"))
        print("Private HTML successfully regenerated at: private_build/kb13_SoManyLeaves/index.html")
except Exception as e:
    print(f"Could not auto-generate Private Story: {e}")

# Step 3a: Package for Public Download
Gather all your standard HTML and audio assets into a single ZIP file.

In [ ]:
### Step 3: Package for Download
# This cell gathers your HTML and audio assets into a single ZIP file for easy downloading.

import zipfile
import os
from google.colab import files as colab_files

def package_complete_story(output_filename="story_web_package.zip"):
    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        found_files = 0
        for root, dirs, files in os.walk('.'):
            if '.config' in root or 'sample_data' in root: continue
            for file in files:
                if file.endswith('.html') or file.endswith('.mp3'):
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, '.')
                    zipf.write(file_path, arcname)
                    found_files += 1

    if found_files > 0:
        print(f"Packaged {found_files} assets into {output_filename}")
        colab_files.download(output_filename)
    else:
        print("No HTML or MP3 files found to package. Please run the conversion first.")

package_btn = widgets.Button(description='Download Package (ZIP)', button_style='success')
package_out = widgets.Output()

def on_pkg_click(b):
    with package_out:
        package_out.clear_output()
        package_complete_story()

package_btn.on_click(on_pkg_click)
display(package_btn, package_out)

# Step 3b: Page for Private Download
Package the HTML and assets generated by the Private Story Converter.

In [ ]:
### Step 3b: Package Private Story (ZIP)

package_btn_p = widgets.Button(description='Download Private Package', button_style='success')
package_out_p = widgets.Output()

def on_pkg_p_click(b):
    with package_out_p:
        package_out_p.clear_output()
        package_complete_story("private_story_package.zip")

package_btn_p.on_click(on_pkg_p_click)
display(package_btn_p, package_out_p)

# Step 4: Asset Cleanup Tool
Use this tool to delete generated folders if you want to start fresh or save disk space.

In [ ]:
### Step 4: Asset Cleanup Tool
# Use this cell to delete generated folders if you want to start fresh or save space.

import shutil

def delete_directory(directory_path):
    if os.path.exists(directory_path) and os.path.isdir(directory_path):
        try:
            shutil.rmtree(directory_path)
            print(f"Successfully deleted: {directory_path}")
        except Exception as e:
            print(f"Error deleting {directory_path}: {e}")
    else:
        print(f"Directory not found: {directory_path}")

delete_target = widgets.Text(placeholder='folder_name', description='Delete Folder:')
delete_btn = widgets.Button(description='Delete Folder', button_style='danger')
delete_out = widgets.Output()

def on_delete_click(b):
    with delete_out:
        delete_out.clear_output()
        if delete_target.value:
            delete_directory(delete_target.value)
        else:
            print("Please enter a folder name (e.g., audio_ka1_CatAndRat)")

delete_btn.on_click(on_delete_click)
display(delete_target, delete_btn, delete_out)